In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")   # Windows RTX
elif torch.backends.mps.is_available():
    device = torch.device("mps")    # Mac M4 Max
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


# Exploring the Data

In [ ]:
from pathlib import Path

import pandas as pd
import pydicom
import matplotlib.pyplot as plt

DATA_ROOT = Path("data/images")
LABEL_FILE = Path("assignment1_labels.csv")
MAPPING_FILE = Path("rsna_to_nih_mapping.csv")

dicom_files = list(DATA_ROOT.rglob("*.dcm"))

labels = pd.read_csv(LABEL_FILE)
mapping = pd.read_csv(MAPPING_FILE)

print("DICOM files:", len(dicom_files))
print("Annotation rows:", len(labels))
print("Unique examinations:", labels["patientId"].nunique())
print("Unique source patients:", mapping["nih_patient_id"].nunique())

display(labels.head())
display(mapping.head())

DICOM files: 25684
Annotation rows: 28989
Unique examinations: 25684
Unique source patients: 11171


,patientId,x,y,width,height,Target,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID
0,1.2.276.0.7230010.3.1.4.8323329.5872.151787431...,NaN,NaN,NaN,NaN,0,1.2.276.0.7230010.3.1.2.8323329.5872.151787431...,1.2.276.0.7230010.3.1.3.8323329.5872.151787431...,1.2.276.0.7230010.3.1.4.8323329.5872.151787431...
1,1.2.276.0.7230010.3.1.4.8323329.15466.15178743...,NaN,NaN,NaN,NaN,0,1.2.276.0.7230010.3.1.2.8323329.15466.15178743...,1.2.276.0.7230010.3.1.3.8323329.15466.15178743...,1.2.276.0.7230010.3.1.4.8323329.15466.15178743...
2,1.2.276.0.7230010.3.1.4.8323329.10973.15178743...,564.0,356.0,274.0,496.0,1,1.2.276.0.7230010.3.1.2.8323329.10973.15178743...,1.2.276.0.7230010.3.1.3.8323329.10973.15178743...,1.2.276.0.7230010.3.1.4.8323329.10973.15178743...
3,1.2.276.0.7230010.3.1.4.8323329.10973.15178743...,195.0,320.0,281.0,515.0,1,1.2.276.0.7230010.3.1.2.8323329.10973.15178743...,1.2.276.0.7230010.3.1.3.8323329.10973.15178743...,1.2.276.0.7230010.3.1.4.8323329.10973.15178743...
4,1.2.276.0.7230010.3.1.4.8323329.22658.15178744...,NaN,NaN,NaN,NaN,0,1.2.276.0.7230010.3.1.2.8323329.22658.15178744...,1.2.276.0.7230010.3.1.3.8323329.22658.15178744...,1.2.276.0.7230010.3.1.4.8323329.22658.15178744...


,patientId,nih_patient_id,nih_image_id
0,1.2.276.0.7230010.3.1.4.8323329.10001.15178743...,11683,00011683_018.png
1,1.2.276.0.7230010.3.1.4.8323329.10002.15178743...,2245,00002245_000.png
2,1.2.276.0.7230010.3.1.4.8323329.10004.15178743...,30355,00030355_000.png
3,1.2.276.0.7230010.3.1.4.8323329.10005.15178743...,14758,00014758_000.png
4,1.2.276.0.7230010.3.1.4.8323329.10006.15178743...,4082,00004082_008.png


In [2]:
print(labels.info())

display(labels.isna().sum())

display(labels["Target"].value_counts())

annotations_per_exam = labels.groupby("patientId").size()

display(annotations_per_exam.value_counts().sort_index())

<class 'pandas.DataFrame'>
RangeIndex: 28989 entries, 0 to 28988
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patientId          28989 non-null  str    
 1   x                  8964 non-null   float64
 2   y                  8964 non-null   float64
 3   width              8964 non-null   float64
 4   height             8964 non-null   float64
 5   Target             28989 non-null  int64  
 6   StudyInstanceUID   28989 non-null  str    
 7   SeriesInstanceUID  28989 non-null  str    
 8   SOPInstanceUID     28989 non-null  str    
dtypes: float64(4), int64(1), str(4)
memory usage: 2.0 MB
None


patientId                0
x                    20025
y                    20025
width                20025
height               20025
Target                   0
StudyInstanceUID         0
SeriesInstanceUID        0
SOPInstanceUID           0
dtype: int64

Target
0    20025
1     8964
Name: count, dtype: int64

1    22506
2     3062
3      105
4       11
Name: count, dtype: int64

In [5]:
exam_labels = (
    labels.groupby("patientId", as_index=False)["Target"]
    .max()
)

print("Examinations:", len(exam_labels))
display(exam_labels["Target"].value_counts())
display(exam_labels["Target"].value_counts(normalize=True))

Examinations: 25684


Target
0    20025
1     5659
Name: count, dtype: int64

Target
0    0.779668
1    0.220332
Name: proportion, dtype: float64

In [ ]:
# annotations quality:

bbox_columns = ["x", "y", "width", "height"]

positive_rows = labels[labels["Target"] == 1]
negative_rows = labels[labels["Target"] == 0]

annotation_checks = pd.Series({
    "Positive rows with complete bounding boxes":
        positive_rows[bbox_columns].notna().all(axis=1).sum(),

    "Negative rows with no bounding box":
        negative_rows[bbox_columns].isna().all(axis=1).sum(),

    "Examinations with repeated annotation rows":
        (annotations_per_exam > 1).sum(),
})

display(annotation_checks)

duplicate_annotations = labels[
    labels["patientId"].duplicated(keep=False)
].sort_values("patientId")

display(duplicate_annotations.head(10))

Positive rows with complete bounding boxes     8964
Negative rows with no bounding box            20025
Examinations with repeated annotation rows     3178
dtype: int64

,patientId,x,y,width,height,Target,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID
481,1.2.276.0.7230010.3.1.4.8323329.10009.15178743...,611.0,515.0,393.0,327.0,1,1.2.276.0.7230010.3.1.2.8323329.10009.15178743...,1.2.276.0.7230010.3.1.3.8323329.10009.15178743...,1.2.276.0.7230010.3.1.4.8323329.10009.15178743...
532,1.2.276.0.7230010.3.1.4.8323329.10009.15178743...,13.0,526.0,422.0,296.0,1,1.2.276.0.7230010.3.1.2.8323329.10009.15178743...,1.2.276.0.7230010.3.1.3.8323329.10009.15178743...,1.2.276.0.7230010.3.1.4.8323329.10009.15178743...
1890,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...,287.0,254.0,82.0,70.0,1,1.2.276.0.7230010.3.1.2.8323329.10028.15178743...,1.2.276.0.7230010.3.1.3.8323329.10028.15178743...,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...
1940,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...,635.0,622.0,103.0,155.0,1,1.2.276.0.7230010.3.1.2.8323329.10028.15178743...,1.2.276.0.7230010.3.1.3.8323329.10028.15178743...,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...
1941,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...,283.0,394.0,133.0,291.0,1,1.2.276.0.7230010.3.1.2.8323329.10028.15178743...,1.2.276.0.7230010.3.1.3.8323329.10028.15178743...,1.2.276.0.7230010.3.1.4.8323329.10028.15178743...
12675,1.2.276.0.7230010.3.1.4.8323329.10029.15178743...,573.0,256.0,189.0,428.0,1,1.2.276.0.7230010.3.1.2.8323329.10029.15178743...,1.2.276.0.7230010.3.1.3.8323329.10029.15178743...,1.2.276.0.7230010.3.1.4.8323329.10029.15178743...
12676,1.2.276.0.7230010.3.1.4.8323329.10029.15178743...,303.0,197.0,207.0,408.0,1,1.2.276.0.7230010.3.1.2.8323329.10029.15178743...,1.2.276.0.7230010.3.1.3.8323329.10029.15178743...,1.2.276.0.7230010.3.1.4.8323329.10029.15178743...
1103,1.2.276.0.7230010.3.1.4.8323329.10034.15178743...,615.0,278.0,332.0,320.0,1,1.2.276.0.7230010.3.1.2.8323329.10034.15178743...,1.2.276.0.7230010.3.1.3.8323329.10034.15178743...,1.2.276.0.7230010.3.1.4.8323329.10034.15178743...
1102,1.2.276.0.7230010.3.1.4.8323329.10034.15178743...,313.0,430.0,211.0,256.0,1,1.2.276.0.7230010.3.1.2.8323329.10034.15178743...,1.2.276.0.7230010.3.1.3.8323329.10034.15178743...,1.2.276.0.7230010.3.1.4.8323329.10034.15178743...
19685,1.2.276.0.7230010.3.1.4.8323329.10040.15178743...,612.0,579.0,274.0,222.0,1,1.2.276.0.7230010.3.1.2.8323329.10040.15178743...,1.2.276.0.7230010.3.1.3.8323329.10040.15178743...,1.2.276.0.7230010.3.1.4.8323329.10040.15178743...


In [8]:
# mapping quality

dataset = exam_labels.merge(
    mapping,
    on="patientId",
    how="left",
    validate="one_to_one",
    indicator=True,
)

display(dataset["_merge"].value_counts())

assert dataset["_merge"].eq("both").all(), "Some examinations have no NIH mapping."

dataset = dataset.drop(columns="_merge")

exams_per_source_patient = (
    dataset.groupby("nih_patient_id")["patientId"]
    .nunique()
)

print("Unique source patients:", dataset["nih_patient_id"].nunique())
print("Patients with multiple examinations:", (exams_per_source_patient > 1).sum())

display(exams_per_source_patient.describe())

_merge
both          25684
left_only         0
right_only        0
Name: count, dtype: int64

Unique source patients: 11171
Patients with multiple examinations: 4198


count    11171.000000
mean         2.299167
std          3.307870
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         63.000000
Name: patientId, dtype: float64

In [10]:
# dicom metadata;

metadata_rows = []

for dcm_path in dicom_files:
    dicom = pydicom.dcmread(
        dcm_path,
        stop_before_pixels=True,
        specific_tags=[
            "Rows",
            "Columns",
            "PhotometricInterpretation",
            "ViewPosition",
            "PatientSex",
            "Modality",
        ],
    )

    metadata_rows.append({
        "patientId": dcm_path.stem,
        "dicom_path": dcm_path,
        "Rows": getattr(dicom, "Rows", None),
        "Columns": getattr(dicom, "Columns", None),
        "PhotometricInterpretation": getattr(
            dicom, "PhotometricInterpretation", None
        ),
        "ViewPosition": getattr(dicom, "ViewPosition", None),
        "PatientSex": getattr(dicom, "PatientSex", None),
        "Modality": getattr(dicom, "Modality", None),
    })

dicom_metadata = pd.DataFrame(metadata_rows)

dataset = dataset.merge(
    dicom_metadata,
    on="patientId",
    how="left",
    validate="one_to_one",
)

for column in [
    "Rows",
    "Columns",
    "PhotometricInterpretation",
    "ViewPosition",
    "PatientSex",
    "Modality",
]:
    print(f"\n{column}")
    display(dataset[column].value_counts(dropna=False))


Rows


Rows
1024    25684
Name: count, dtype: int64


Columns


Columns
1024    25684
Name: count, dtype: int64


PhotometricInterpretation


...

ViewPosition
PA    13979
AP    11705
Name: count, dtype: int64


PatientSex


PatientSex
M    14593
F    11091
Name: count, dtype: int64


Modality


Modality
CR    25684
Name: count, dtype: int64